## Importing Libraries and Setting Pandas Features

In [28]:
# import all necessary libraries
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import time
import random
import os
import re
from pathlib import Path

In [29]:
# set pandas display setting
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)

## Cricket Dataframe

In [30]:
# define url and headers for reference later
url = 'https://orthsoc.org/sina/cricklist.htm'
headers = {
    "User-Agent": (
        "AcademicResearchBot"
        "(collecting data for non-commercial communication signals research project)"
    ),
    "Accept": "text/html,application/xhtml+xml",
    "Accept-Language": "en-US,en;q=0.9",
    "Connection": "keep-alive"
}

In [31]:
# get code for cricket webpage
response = requests.get(url, headers = headers)
soup = BeautifulSoup(response.text, 'html.parser')

In [32]:
# retrieve species names and urls and store them in species_list
cricket_species_list = []
for species in soup.find_all("h3", class_="species"):
    a_tag = species.find("a")
    if a_tag:
        species_name = a_tag.get_text(strip = True)
        species_url = urljoin("https://orthsoc.org/sina/", a_tag["href"])
        cricket_species_list.append({"species": species_name, "url": species_url})

In [33]:
grouped_data = []
session = requests.Session()
session.headers.update(headers)

for species in cricket_species_list:
    r = session.get(species["url"], timeout=15)
    r.encoding = r.apparent_encoding
    soup = BeautifulSoup(r.text, "html.parser")

    species_name = species["species"]

    # finding range map of species (if possible)
    map_large = None
    map_figure = soup.find("table", class_="images")

    if map_figure:
        map_link = map_figure.find("a", href=True)
        if map_link:
            href = map_link["href"]

            match = re.search(r"(\d+)m\.htm", href, re.I)
            if match:
                code = match.group(1)
                href = f"{code}mc.gif"
                map_large = urljoin(species["url"], href)

    if not map_large:
        for row in soup.find_all("tr"):
            caps = row.find_all("td", class_="captions")
            for idx, cap in enumerate(caps):
                if "map" in cap.get_text(" ", strip=True).lower():
                    prev = row.find_previous_sibling("tr")
                    if not prev:
                        continue

                    imgs = prev.find_all("td", class_="images")
                    if idx >= len(imgs):
                        continue

                    link = imgs[idx].find("a", href=True)
                    if not link:
                        continue

                    href = link["href"]
                    match = re.search(r"(\d+)m\.htm", href, re.I)
                    if match:
                        code = match.group(1)
                        href = f"{code}mc.gif"
                        map_large = urljoin(species["url"], href)
                    break
            if map_large:
                break

    recording_blocks = soup.find_all(
        "div",
        class_=lambda x: x and "recording" in x
    )

    for rec in recording_blocks:

        text = rec.get_text(" ", strip=True)

        # find audio file
        audio = None
        audio_tag = rec.find("audio")
        if audio_tag:
            source = audio_tag.find("source", src=True)
            if source:
                audio = urljoin(species["url"], source["src"])

        # find spectrogram
        spectrogram = None

        for img in rec.find_all("img", src=True):
            src = img["src"].lower()
            alt = (img.get("alt") or "").lower()

            combined = f"{src} {alt} {text.lower()}"

            if any(word in combined for word in [
                "spectrogram",
                "sonogram",
                "waveform",
                "graph",
                "graphed song"
            ]):
                spectrogram = urljoin(species["url"], img["src"])
                break

        # skip if nothing useful found
        if not spectrogram:
            continue

        # attempt to extract temperature from description
        temp = None
        if "°" in text:
            i = text.find("°")
            temp = text[max(0, i - 5): i + 1].strip()

        # attempt to extraction location from description
        loc = "No location found"

        if "from" in text:
            start = text.find("from") + 5
            end = text.find("°") if "°" in text else len(text)
            loc = text[start:end]

        elif "in" in text:
            start = text.find("in") + 3
            end = text.find("°") if "°" in text else len(text)
            loc = text[start:end]

        loc = loc.strip(" ,;.")

        # put all pieces of data in grouped_data
        grouped_data.append({
            "Species": species_name,
            "URL": species["url"],
            "Spectrogram": spectrogram,
            "Audio": audio,
            "Description": text,
            "Temperature": temp,
            "Location": loc,
            "Map": map_large
        })

    time.sleep(random.uniform(1, 2))

In [34]:
cricket_df = pd.DataFrame(grouped_data)

In [35]:
cricket_df.columns = ['Species', 'URL', 'Spectrogram', 'Audio_Link', 'Description of Whole Audio File', 'Temperature (°C)', 'Location', 'Map']

In [36]:
cricket_df.head()

,Species,URL,Spectrogram,Audio_Link,Description of Whole Audio File,Temperature (°C),Location,Map
0,Gryllotalpa gryllotalpa,https://orthsoc.org/sina/363a.htm,https://orthsoc.org/sina/363ss2.jpg,https://orthsoc.org/sina/media/363ss2.mp3,This sound spectrogram is a 2 s excerpt of the 19 s audio file accesssible above. The excerpt begins at 2 s.,None,at 2 s,https://orthsoc.org/sina/363mc.gif
1,Gryllotalpa major,https://orthsoc.org/sina/361a.htm,https://orthsoc.org/sina/361ss2.gif,https://orthsoc.org/sina/media/361ss2.mp3,This sound spectrogram is a 2 s excerpt of the 20 s audio file accesssible above.,None,No location found,https://orthsoc.org/sina/361mc.gif
2,Neocurtilla hexadactyla,https://orthsoc.org/sina/351a.htm,https://orthsoc.org/sina/351ss2.gif,https://orthsoc.org/sina/media/351ss2.mp3,This spectrogram is a 2 s excerpt of the 16 s audio file accessible above.,None,No location found,https://orthsoc.org/sina/351mc.gif
3,Neoscapteriscus borellii,https://orthsoc.org/sina/341a.htm,https://orthsoc.org/sina/341ss2.gif,https://orthsoc.org/sina/media/341ss2.mp3,This spectrogram is a 2 s excerpt of the 20 s audio file accessible above.,None,No location found,https://orthsoc.org/sina/341mc.gif
4,Neoscapteriscus vicinus,https://orthsoc.org/sina/342a.htm,https://orthsoc.org/sina/342ss2.gif,https://orthsoc.org/sina/media/342ss2.mp3,This spectrogram is a 1 s excerpt of the 22 s audio file accessible above. Differences among pulses are a result of limitations of the software used to produce the spectrogram. Click on spectrogram to expand its spectrographic image.,None,No location found,https://orthsoc.org/sina/342mc.gif


## Katydid Dataframe

In [37]:
# define for reference later
url = 'https://orthsoc.org/sina/katylist.htm'

In [38]:
# get code for katydid webpage
response = requests.get(url, headers = headers)
katydid_soup = BeautifulSoup(response.text, 'html.parser')

In [39]:
# retrieve species names and urls and store them in species_list
katydid_species_list = []
for species in katydid_soup.find_all("h3", class_="species"):
    a_tag = species.find("a")
    if a_tag:
        species_name = a_tag.get_text(strip = True)
        species_url = urljoin("https://orthsoc.org/sina/", a_tag["href"])
        katydid_species_list.append({"species": species_name, "url": species_url})

In [40]:
grouped_data = []
session = requests.Session()
session.headers.update(headers)

for species in katydid_species_list:
    r = session.get(species["url"], timeout=15)
    r.encoding = r.apparent_encoding
    soup = BeautifulSoup(r.text, "html.parser")

    species_name = species["species"]

    # finding range map of species (if possible)
    map_large = None
    map_figure = soup.find("table", class_="images")

    if map_figure:
        map_link = map_figure.find("a", href=True)
        if map_link:
            href = map_link["href"]

            match = re.search(r"(\d+)m\.htm", href, re.I)
            if match:
                code = match.group(1)
                href = f"{code}mc.gif"
                map_large = urljoin(species["url"], href)

    if not map_large:
        for row in soup.find_all("tr"):
            caps = row.find_all("td", class_="captions")
            for idx, cap in enumerate(caps):
                if "map" in cap.get_text(" ", strip=True).lower():
                    prev = row.find_previous_sibling("tr")
                    if not prev:
                        continue

                    imgs = prev.find_all("td", class_="images")
                    if idx >= len(imgs):
                        continue

                    link = imgs[idx].find("a", href=True)
                    if not link:
                        continue

                    href = link["href"]
                    match = re.search(r"(\d+)m\.htm", href, re.I)
                    if match:
                        code = match.group(1)
                        href = f"{code}mc.gif"
                        map_large = urljoin(species["url"], href)
                    break
            if map_large:
                break

    recording_blocks = soup.find_all(
        "div",
        class_=lambda x: x and "recording" in x
    )

    for rec in recording_blocks:

        text = rec.get_text(" ", strip=True)

        # find audio file
        audio = None
        audio_tag = rec.find("audio")
        if audio_tag:
            source = audio_tag.find("source", src=True)
            if source:
                audio = urljoin(species["url"], source["src"])

        # find spectrogram
        spectrogram = None

        for img in rec.find_all("img", src=True):
            src = img["src"].lower()
            alt = (img.get("alt") or "").lower()

            combined = f"{src} {alt} {text.lower()}"

            if any(word in combined for word in [
                "spectrogram",
                "sonogram",
                "waveform",
                "graph",
                "graphed song"
            ]):
                spectrogram = urljoin(species["url"], img["src"])
                break

        # skip if nothing useful found
        if not spectrogram:
            continue

        # attempt to extract temperature from description
        temp = None
        if "°" in text:
            i = text.find("°")
            temp = text[max(0, i - 5): i + 1].strip()

        # attempt to extraction location from description
        loc = "No location found"

        if "from" in text:
            start = text.find("from") + 5
            end = text.find("°") if "°" in text else len(text)
            loc = text[start:end]

        elif "in" in text:
            start = text.find("in") + 3
            end = text.find("°") if "°" in text else len(text)
            loc = text[start:end]

        loc = loc.strip(" ,;.")

        # put all pieces of data in grouped_data
        grouped_data.append({
            "Species": species_name,
            "URL": species["url"],
            "Spectrogram": spectrogram,
            "Audio": audio,
            "Description": text,
            "Temperature": temp,
            "Location": loc,
            "Map": map_large
        })

    time.sleep(random.uniform(1, 2))

In [41]:
katydid_df = pd.DataFrame(grouped_data)

In [42]:
katydid_df.columns = ['Species', 'URL', 'Spectrogram', 'Audio_Link', 'Description', 'Temperature (°C)', 'Location', 'Map']

In [43]:
katydid_df.head()

,Species,URL,Spectrogram,Audio_Link,Description,Temperature (°C),Location,Map
0,Cyphoderris buckelli,https://orthsoc.org/sina/337a.htm,https://orthsoc.org/sina/337ss2.gif,https://orthsoc.org/sina/media/337ss2.mp3,This waveform is a 2 s excerpt of the 10 s audio file accessible above (slowed to half speed). Click on waveform to expand last 0.25 s of waveform.,None,No location found,https://orthsoc.org/sina/337mc.gif
1,Conocephalus aigialus,https://orthsoc.org/sina/230a.htm,https://orthsoc.org/sina/230ss2.gif,https://orthsoc.org/sina/media/230ss2.mp3,This waveform is a 5 s excerpt of the 20 s audio file accessible above. Click on waveform to expand last 0.5 s of song.,None,No location found,https://orthsoc.org/sina/230mc.gif
2,Conocephalus allardi,https://orthsoc.org/sina/220a.htm,https://orthsoc.org/sina/220ss2.gif,https://orthsoc.org/sina/media/220ss2.mp3,This waveform is a 5 s excerpt of the 70 s audio file accessible above. Click on waveform to expand 0.5 s of the 5 s song.,None,No location found,https://orthsoc.org/sina/220mc.gif
3,Conocephalus attenuatus,https://orthsoc.org/sina/228a.htm,https://orthsoc.org/sina/228ss2.gif,https://orthsoc.org/sina/media/228ss2.mp3,This waveform is a 5 s excerpt of the 22 s audio file accessible above. Click on waveform to expand last 0.5 s of song.,None,No location found,https://orthsoc.org/sina/228mc.gif
4,Conocephalus brevipennis,https://orthsoc.org/sina/234a.htm,https://orthsoc.org/sina/234ss2.gif,https://orthsoc.org/sina/media/234ss2.mp3,This waveform is a 5 s excerpt of the 20 s audio file accessible above. Click on waveform to expand 0.5 s of buzz.,None,No location found,https://orthsoc.org/sina/234mc.gif


## Frog Dataframe

In [44]:
# extract the data from different pages by looping over all frog pages in the API
page_num = 1
base_url = f'https://xeno-canto.org/api/3/recordings?query=grp:frogs&page={page_num}&key=18a98cdc0e5e1df2c91e00a17f4bfcf89cd501b6'
recordings = []
response = requests.get(base_url)
data = response.json()
recordings.extend(data["recordings"])
for n in range(2,58):
    page_num = n
    response = requests.get(base_url)
    data = response.json()
    recordings.extend(data["recordings"])
    time.sleep(random.uniform(2,4))

In [45]:
# flatten nested data
df = pd.json_normalize(recordings)

In [46]:
# extract useful columns
frog_df = df[['gen', 'sp', 'cnt', 'loc', 'lat', 'lon', 'type', 'file', 'sono.med']]

In [47]:
# rename extracted columns
frog_df.columns = ['Genus', 'Species', 'Country', 'Location', 'Latitude', 'Longitude', 'Call Type', 'File', 'Spectrogram']

In [48]:
frog_df.head()

,Genus,Species,Country,Location,Latitude,Longitude,Call Type,File,Spectrogram
0,Hyla,meridionalis,France,"Arrondissement de Marseille (near Marseille), Bouches-du-Rhone, Provence-Alpes-Côte d'Azur",43.2755,5.48,territorial call,https://xeno-canto.org/1139790/download,https://xeno-canto.org/sounds/spectrograms/YEUMVPMDCG/1139790/grey-medium.png
1,Hyla,meridionalis,France,"Arrondissement de Marseille (near Marseille), Bouches-du-Rhone, Provence-Alpes-Côte d'Azur",43.2755,5.48,territorial call,https://xeno-canto.org/1139789/download,https://xeno-canto.org/sounds/spectrograms/YEUMVPMDCG/1139789/grey-medium.png
2,Hyla,meridionalis,France,"Arrondissement de Marseille (near Marseille), Bouches-du-Rhone, Provence-Alpes-Côte d'Azur",43.2755,5.48,territorial call,https://xeno-canto.org/1139788/download,https://xeno-canto.org/sounds/spectrograms/YEUMVPMDCG/1139788/grey-medium.png
3,Phyllomedusa,distincta,Brazil,"Estação Ecológica Juréia-Itatins, Peruíbe, São Paulo",-24.3893,-47.0188,advertisement call,https://xeno-canto.org/1139730/download,https://xeno-canto.org/sounds/spectrograms/YOXWKYFZAP/1139730/grey-medium.png
4,Odontophrynus,asper,Paraguay,"Potrero Jakarey, Valenzuela, Cordillera",-25.6313,-56.8743,,https://xeno-canto.org/1139604/download,https://xeno-canto.org/sounds/spectrograms/VALRNZCFXZ/1139604/grey-medium.png


## Downloading Files

### Crickets

In [49]:
def get_file_id(url):
    return os.path.splitext(os.path.basename(str(url)))[0]

In [50]:
base_folder = Path.home() / "Discrete_Signals"

In [51]:
base_folder = Path.home() / "Discrete_Signals"

group_folders = {
    "crickets": base_folder / "Crickets",
    "katydids": base_folder / "Katydids",
    "frogs": base_folder / "Frogs"
}

for folder in group_folders.values():
    folder.mkdir(parents=True, exist_ok=True)

In [52]:
def download_file(url, path, headers):
    try:
        r = requests.get(url, headers=headers, timeout=20)
        r.raise_for_status()

        content_type = r.headers.get("Content-Type", "").lower()

        if "text/html" in content_type:
            print(f"Skipping HTML page masquerading as asset: {url}")
            return

        # making sure it's not an error page
        if r.content.startswith(b"<!DOCTYPE") or r.content.startswith(b"<html"):
            print(f"Skipping text/html payload for: {url}")
            return

        with open(path, "wb") as f:
            f.write(r.content)

    except Exception as e:
        print(f"Download failed: {url}")
        print(e)

In [53]:
for idx, row in cricket_df.iterrows():
    spectrogram_url = row.get("Spectrogram")
    audio_url = row.get("Audio_Link")
    map_url = row.get("Map")

    if pd.isna(spectrogram_url) and pd.isna(audio_url) and pd.isna(map_url):
        continue

    species = str(row["Species"]).replace(" ", "_")

    # create species subfolder inside Crickets
    species_folder = group_folders["crickets"] / species
    species_folder.mkdir(parents=True, exist_ok=True)

    spec_id = None

    if pd.notna(spectrogram_url) and spectrogram_url:
        ext = os.path.splitext(spectrogram_url)[1].lower()
        if ext not in [".jpg", ".jpeg", ".png", ".gif", ".webp"]:
            ext = ".gif"

        spec_id = get_file_id(spectrogram_url)
        spec_path = species_folder / f"{species}_spectrogram_{spec_id}{ext}"
        download_file(spectrogram_url, spec_path, headers)

    if pd.notna(audio_url):
        audio_url = str(audio_url).strip()
        ext = os.path.splitext(audio_url)[1].lower()
        if ext not in [".mp3", ".wav", ".ogg"]:
            ext = ".wav"

        audio_id = spec_id or get_file_id(audio_url)
        audio_path = species_folder / f"{species}_audio_{audio_id}{ext}"

        try:
            r = session.get(audio_url, timeout=20)
            r.raise_for_status()

            with open(audio_path, "wb") as f:
                f.write(r.content)

        except Exception as e:
            print(f"Error downloading audio {audio_url}")
            print(e)

    if pd.notna(map_url) and map_url:
        ext = os.path.splitext(map_url)[1].lower()
        if ext not in [".gif", ".jpg", ".jpeg", ".png", ".webp"]:
            ext = ".gif"

        map_id = get_file_id(map_url)
        map_path = species_folder / f"{species}_map_{map_id}{ext}"
        download_file(map_url, map_path, headers)

Download failed: https://orthsoc.org/sina/469mc.gif
404 Client Error: Not Found for url: https://orthsoc.org/sina/469mc.gif
Download failed: https://orthsoc.org/sina/469mc.gif
404 Client Error: Not Found for url: https://orthsoc.org/sina/469mc.gif
Download failed: https://orthsoc.org/sina/469mc.gif
404 Client Error: Not Found for url: https://orthsoc.org/sina/469mc.gif
Download failed: https://orthsoc.org/sina/721mc.gif
404 Client Error: Not Found for url: https://orthsoc.org/sina/721mc.gif
Download failed: https://orthsoc.org/sina/721mc.gif
404 Client Error: Not Found for url: https://orthsoc.org/sina/721mc.gif
Download failed: https://orthsoc.org/sina/722mc.gif
404 Client Error: Not Found for url: https://orthsoc.org/sina/722mc.gif
Download failed: https://orthsoc.org/sina/722mc.gif
404 Client Error: Not Found for url: https://orthsoc.org/sina/722mc.gif
Download failed: https://orthsoc.org/sina/723mc.gif
404 Client Error: Not Found for url: https://orthsoc.org/sina/723mc.gif
Download

### Katydids

In [54]:
for idx, row in katydid_df.iterrows():
    spectrogram_url = row.get("Spectrogram")
    audio_url = row.get("Audio_Link")
    map_url = row.get("Map")

    if pd.isna(spectrogram_url) and pd.isna(audio_url) and pd.isna(map_url):
        continue

    species = str(row["Species"]).replace(" ", "_")

    species_folder = group_folders["katydids"] / species
    species_folder.mkdir(parents=True, exist_ok=True)

    spec_id = None

    if pd.notna(spectrogram_url) and spectrogram_url:
        ext = os.path.splitext(spectrogram_url)[1].lower()
        if ext not in [".jpg", ".jpeg", ".png", ".gif", ".webp"]:
            ext = ".gif"

        spec_id = get_file_id(spectrogram_url)
        spec_path = species_folder / f"{species}_spectrogram_{spec_id}{ext}"
        download_file(spectrogram_url, spec_path, headers)

    if pd.notna(audio_url):
        audio_url = str(audio_url).strip()
        ext = os.path.splitext(audio_url)[1].lower()
        if ext not in [".mp3", ".wav", ".ogg"]:
            ext = ".wav"

        audio_id = spec_id or get_file_id(audio_url)
        audio_path = species_folder / f"{species}_audio_{audio_id}{ext}"

        try:
            r = session.get(audio_url, timeout=20)
            r.raise_for_status()

            with open(audio_path, "wb") as f:
                f.write(r.content)

        except Exception as e:
            print(f"Error downloading audio {audio_url}")
            print(e)

    if pd.notna(map_url) and map_url:
        ext = os.path.splitext(map_url)[1].lower()
        if ext not in [".gif", ".jpg", ".jpeg", ".png", ".webp"]:
            ext = ".gif"

        map_id = get_file_id(map_url)
        map_path = species_folder / f"{species}_map_{map_id}{ext}"
        download_file(map_url, map_path, headers)

Download failed: https://orthsoc.org/sina/235mc.gif
404 Client Error: Not Found for url: https://orthsoc.org/sina/235mc.gif
Download failed: https://orthsoc.org/sina/801mc.gif
404 Client Error: Not Found for url: https://orthsoc.org/sina/801mc.gif
Download failed: https://orthsoc.org/sina/801mc.gif
404 Client Error: Not Found for url: https://orthsoc.org/sina/801mc.gif
Download failed: https://orthsoc.org/sina/802mc.gif
404 Client Error: Not Found for url: https://orthsoc.org/sina/802mc.gif
Download failed: https://orthsoc.org/sina/802mc.gif
404 Client Error: Not Found for url: https://orthsoc.org/sina/802mc.gif
Download failed: https://orthsoc.org/sina/803mc.gif
404 Client Error: Not Found for url: https://orthsoc.org/sina/803mc.gif
Download failed: https://orthsoc.org/sina/803mc.gif
404 Client Error: Not Found for url: https://orthsoc.org/sina/803mc.gif
Download failed: https://orthsoc.org/sina/804mc.gif
404 Client Error: Not Found for url: https://orthsoc.org/sina/804mc.gif
Download

### Frogs

In [55]:
def get_id(url):
    return os.path.splitext(os.path.basename(url))[0]

def safe_download(session, url, path):
    if path.exists():
        return

    r = session.get(url, timeout=20)
    if r.status_code == 200:
        with open(path, "wb") as f:
            f.write(r.content)
    else:
        print(f"Failed ({r.status_code}): {url}")

In [56]:
def download_frogs(frog_df, base_folder, headers):
    frog_root = base_folder / "Frogs"
    session = requests.Session()
    session.headers.update(headers)
    n = 1
    for idx, row in frog_df.iterrows():

        genus = str(row["Genus"])
        species = str(row["Species"]).strip("'")
        species_name = f"{genus}_{species}"

        audio_url = row.get("File")
        spectrogram_url = row.get("Spectrogram")

        # skip empty rows
        if pd.isna(audio_url) and pd.isna(spectrogram_url):
            continue

        # create folder
        species_folder = frog_root / species_name
        species_folder.mkdir(parents=True, exist_ok=True)

        if pd.notna(audio_url):
            ext = os.path.splitext(str(audio_url))[1].lower()
            if ext not in [".mp3", ".wav", ".ogg"]:
                ext = ".mp3"

            audio_id = get_file_id(audio_url)
            audio_path = species_folder / f"{species}_audio_{audio_id}{ext}"

            try:
                if not audio_path.exists():
                    r = session.get(audio_url, timeout=20)
                    if r.status_code == 200:
                        with open(audio_path, "wb") as f:
                            f.write(r.content)
                    else:
                        print(f"Audio failed ({r.status_code}): {audio_url}")
            except Exception as e:
                print(f"Error downloading audio: {audio_url}")
                print(e)
                
        if pd.notna(spectrogram_url):
            ext = os.path.splitext(str(spectrogram_url))[1].lower()
            if ext not in [".jpg", ".jpeg", ".png", ".gif", ".webp"]:
                ext = ".png"

            spec_id = get_file_id(spectrogram_url)
            spec_path = species_folder / f"{species}_spectrogram_{spec_id}{ext}"

            try:
                if not spec_path.exists():
                    r = session.get(spectrogram_url, timeout=20)
                    if r.status_code == 200:
                        with open(spec_path, "wb") as f:
                            f.write(r.content)
                    else:
                        print(f"Spectrogram failed ({r.status_code}): {spectrogram_url}")
            except Exception as e:
                print(f"Error downloading spectrogram: {spectrogram_url}")
                print(e)

        n += 1
        print(f'{species_name} downloaded, {n*100/851}% completed')
        time.sleep(random.uniform(1,2))

In [57]:
download_frogs(frog_df, base_folder, headers)

Hyla_meridionalis downloaded, 0.23501762632197415% completed
Hyla_meridionalis downloaded, 0.3525264394829612% completed
Hyla_meridionalis downloaded, 0.4700352526439483% completed
Phyllomedusa_distincta downloaded, 0.5875440658049353% completed
Odontophrynus_asper downloaded, 0.7050528789659224% completed
Pelophylax_esculentus downloaded, 0.8225616921269095% completed
Melanophryniscus_fulvoguttatus downloaded, 0.9400705052878966% completed
Hylarana_baramica downloaded, 1.0575793184488838% completed
Pelophylax_ridibundus downloaded, 1.1750881316098707% completed
Pelophylax_ridibundus downloaded, 1.2925969447708578% completed
Pelophylax_ridibundus downloaded, 1.4101057579318448% completed
Hyla_arborea downloaded, 1.527614571092832% completed
Hyla_arborea downloaded, 1.645123384253819% completed
Hyla_arborea downloaded, 1.762632197414806% completed
Bombina_bombina downloaded, 1.8801410105757932% completed
Eleutherodactylus_cystignathoides downloaded, 1.9976498237367804% completed
Strauch